# Question 1: SMB Sellers Identification

## Assignment
> In our businesses we have always seen it as a strength that registration is very low effort. You only need an email address and a phone number to start trading on our platforms.
>
> The downside of this is that we do not have a full customer view of our users. Especially in our SYI flow, we do not know much about our sellers. We do, however, know a lot about what happens on our platforms. We can easily understand the behavior of our visitors; buyers, sellers, and browsers.
>
> In the first attachment, a list of information can be found that we do have on our sellers and the activity. This data goes three years back.
>
> It is December 2024. We want to introduce SMB bundles, which will make it easier for SMB sellers to use our platform to reach our enormous customer base. To launch this, a big task is to identify which sellers are actual business sellers and which are consumers.
>
> We cannot simply call all 10+ million users to ask this, and using a pop-up on our platform is not expected to bring reliable results. We need to narrow down this base as good as we can using the existing data.
>
> Using the description of the available data, think of identifiers that point towards a seller being SMB, and argue why you believe these could be good indicators. Combine information from the various sources to think of 3-8 good identifiers. (For example, “Every seller that has listed at least 2 advertisements in the category ‘Tuin en Terras’ (Garden and terrace) is likely to be a business seller”.)




## 1. SMB Identification

We can infer SMB likelihood from seller behavior, paid intent, category focus, and buyer demand. The goal is a ranked seller list, not a hard SMB/consumer label.

Use the available schemas to create seller-level indicators. Each identification category is a separate signal; the final score combines them.





### 1.1 Direct Business Markers

**Signal:** fields that directly indicate business setup or off-platform commercial presence.

Schemas and columns:

| Schema | Columns used |
| --- | --- |
| **User Related Info** | `User id`, `Bank account is corporate y/n` |
| **Ad Related Info** | `User id`, `External URL attached to the ad y/n` |

Use the latest available corporate bank flag. Measure external URL usage over the last 12 months.

Useful metrics:

| Metric | Columns used | Why useful |
| --- | --- | --- |
| Corporate bank account flag | `User Related Info.Bank account is corporate y/n` | Direct business setup signal. |
| External URL attached across multiple ads | `Ad Related Info.User id`, `External URL attached to the ad y/n` in last 12 months | Suggests off-platform business presence or lead routing. |

Candidate rule: latest corporate bank account flag, or external URL usage across multiple ads in the last 12 months.



### 1.2 Paid Usage

**Signal:** sellers who repeatedly pay for visibility, Pro traffic, or invoiced services are more likely to be commercial sellers.

Schemas and columns:

| Schema | Columns used |
| --- | --- |
| **Ad Related Info** | `User id`, `Ad id`, `Insertion fee`, `Ad start date/time`, `Ad end date/time` |
| **Feature Related Info** | `Ad id`, `Feature type`, `Feature start date/time`, `Feature end date/time`, `Feature fee` |
| **Pro Info** | `User id`, `Ad id`, `CPC`, `Daily number of clicks` |
| **Pro Invoicing Info** | `User id`, `Invoice month`, `Total invoice amount`, `Invoice paid date/time` |

Use the last 12 months as the scoring window.

Useful metrics:

| Metric | Columns used | Why useful |
| --- | --- | --- |
| Paid feature usage | `Feature type` in last 12 months | Captures repeated use of paid visibility products. |
| Paid feature spend | `Feature fee` in last 12 months | Measures willingness to pay for visibility. |
| Paid listing fees | `Insertion fee` in last 12 months | Captures paid listing behavior outside feature purchases. |
| Pro CPC usage | `CPC`, `Ad id` in last 12 months | Indicates performance-oriented selling. |
| Pro paid traffic scale | `CPC`, `Daily number of clicks` in last 12 months | Estimates the scale of paid Pro traffic. |
| Invoice months | `Invoice month` in last 12 months | Captures recurring billing, not just one-off payments. |
| Invoice amount | `Total invoice amount` in last 12 months | Captures scale of invoiced spend. |

Candidate rule: repeated paid feature usage, paid feature spend, insertion-fee spend, Pro CPC usage, paid Pro traffic scale, or recurring invoice months in the last 12 months.



### 1.3 General Seller Activity

**Signal:** sellers that repeatedly list and stay active look more business-like than one-off sellers.

Schemas and columns:

| Schema | Columns used |
| --- | --- |
| **User Related Info** | `User id` |
| **Ad Related Info** | `User id`, `Ad id`, `Ad start date/time`, `Ad end date/time` |

Use the last 12 months as the scoring window.

Useful metrics:

| Metric | Columns used | Why useful |
| --- | --- | --- |
| Ads in last 12 months | `Ad Related Info.User id`, `Ad id`, `Ad start date/time` | Captures seller volume. |
| Active months | `Ad start date/time` in last 12 months | Captures persistence. |
| Max concurrent live ads | `Ad start date/time`, `Ad end date/time` in last 12 months | Captures maintained inventory. |
| Recent listing activity | `Ad start date/time` in last 3 months | Keeps outreach focused on reachable sellers. |

Candidate rule: listings in 3+ months, 10+ ads in 12 months or multiple concurrent live ads, and at least one recent listing.




### 1.4 Category Specialization

**Signal:** SMBs often show repeated supply within the same category or subcategory.

Schemas and columns:

| Schema | Columns used |
| --- | --- |
| **Ad Related Info** | `User id`, `Ad id`, `Ad category` |
| **Reference: Categories** | `Category id`, `Category name` |

Use both the main category and the more specific subcategory. Measure specialization in the last 12 months.

Useful metrics:

| Metric | Columns used | Why useful |
| --- | --- | --- |
| Category-adjusted repeated listings | `Ad Related Info.User id`, `Ad category` in last 12 months | Repeated listings are more meaningful in some categories than others. |

Threshold examples:

| Category type | Example categories | Suggested threshold logic |
| --- | --- | --- |
| High-value / dealer-like | Cars; Motorcycles; Caravans and Camping; Water Sports and Boats; Car Parts | Lower threshold: a few listings can already suggest commercial supply. |
| Service / business | Business Goods; Services and Tradespeople; Jobs | Lower threshold: the category itself is closer to business activity. |
| Consumer resale | Clothing - Women; Clothing - Men; Books; Children and Babies; CDs and DVDs; Games | Higher threshold: several listings can still be normal household selling. |
| Broad / mixed | Miscellaneous; Home and Furnishings; Garden and Terrace; Hobby and Leisure; Collectibles | Higher threshold or require another signal, because category alone is noisy. |

Candidate rule: apply category-specific thresholds for repeated listings in the same category/subcategory. Use lower thresholds for high-value, service, or dealer-like categories, and higher thresholds for common consumer resale or broad mixed categories.



### 1.5 Buyer Engagement Aggregated to Seller Level

**Signal:** one popular ad can be consumer behavior; repeated buyer engagement across ads is more SMB-like.

Schemas and columns:

| Schema | Columns used |
| --- | --- |
| **Ad Related Info** | `User id`, `Ad id`, `Ad category` |
| **Messaging** | `Ad id`, `Seller id`, `Buyer id`, `Message date/time` |
| **Traffic Info** | `Page`, `Click`, `User id`, `Specific dimensions per page type` |
| **Pro Info** | `Ad id`, `Daily number of impressions` where available |

Use the last 12 months as the scoring window.

Useful metrics:

| Metric | Columns used | Why useful |
| --- | --- | --- |
| Engaged ads | `Messaging.Ad id`, `Traffic Info.Page`, `Traffic Info.Click` in last 12 months | Avoids one-ad outliers. |
| Unique engaged buyers | `Messaging.Buyer id`, `Messaging.Seller id` in last 12 months | Captures breadth of demand. |
| Months with engagement | `Messaging.Message date/time`, traffic event date/time if available, in last 12 months | Captures persistence. |
| Engagement rate | `Traffic Info.Page`, `Traffic Info.Click`, `Pro Info.Daily number of impressions` where available, in last 12 months | Controls for exposure where possible. |

Candidate rule: top-decile engagement within category mix, 5+ engaged ads in 12 months, engagement in 3+ months, and multiple unique buyers.




## 2. Decision Mechanism

Create a ranked seller list by scoring individual SMB signals. Identification categories organize the signals, but the final score is built from the signals themselves.



### 2.1 SMB Score

Each seller-level signal contributes points based on strength. Identification categories organize the signals, but the score is built from the signals themselves.

| Weight | General rule | Example signals |
| --- | --- | --- |
| Highest | Direct evidence of business setup. | Corporate bank account flag. |
| High | Clear paid selling behavior. | Recurring invoice months, Pro CPC usage, paid feature spend. |
| Lower | SMB-like behavior that is useful but noisier. | High listing volume, category-adjusted repeated listings, buyer engagement across ads. |

Output: ranked sellers with `user_id`, `smb_score`, confidence tier, top reasons, and recommended action.



### 2.2 If True SMB Labels Exist

If true SMB labels become available, replace manual weights with a supervised model using the same seller-level features. Keep the output explainable so stakeholders can see why each seller was prioritized.

